# Model Comparison
Trains Logistic Regression and Random Forest on a 300,000 row sample of the cleaned CICIDS2017 data, then compares them.

In [1]:
import pandas as pd
data = pd.read_csv("../data/processed/cicids2017_clean.csv")
print(data.shape)

(2827876, 79)


## Prepare the data
The full dataset is too large for an 8GB laptop, so we train on a random sample of 300,000 rows. Features are scaled before training.

In [1]:
data = data.sample(n=300000, random_state=42)
print(data.shape)

NameError: name 'data' is not defined

In [ ]:
X = data.drop(columns=['Label'])
y = data['Label']
print(X.shape)
print(y.shape)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape)
print(X_test.shape)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## Baseline: Logistic Regression
A simple model to compare against. It catches most attacks but raises many false alarms.

In [ ]:
from sklearn.linear_model import LogisticRegression
model_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
model_lr.fit(X_train_scaled, y_train)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model_lr.predict(X_test_scaled)
print(classification_report(y_test, y_pred))

## Main model: Random Forest
Much better precision and recall on almost every class. This is the model we saved.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model_rf = RandomForestClassifier(
    n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
)
model_rf.fit(X_train_scaled, y_train)

In [ ]:
y_pred_rf = model_rf.predict(X_test_scaled)
print(classification_report(y_test, y_pred_rf))

## Feature importance
Destination Port is the most useful feature, followed by packet size and timing features.

In [ ]:
importances = pd.Series(model_rf.feature_importances_, index=X.columns)
importances = importances.sort_values(ascending=False)
print(importances.head(15))

## Tuning attempt
Tried 200 trees and max depth 30. Results were nearly the same, so the weak classes (Bot, XSS, Brute Force) are a data problem, not a tuning problem.

In [ ]:
model_rf2 = RandomForestClassifier(
    n_estimators=200, max_depth=30, class_weight='balanced', random_state=42, n_jobs=-1
)
model_rf2.fit(X_train_scaled, y_train)

In [ ]:
y_pred_rf2 = model_rf2.predict(X_test_scaled)
print(classification_report(y_test, y_pred_rf2))

## Save the model
The original Random Forest and the scaler are saved for DevOps to use in predict.py.

In [ ]:
import joblib
joblib.dump(model_rf, "../src/models/random_forest_model.pkl")
joblib.dump(scaler, "../src/models/scaler.pkl")